# Cell Surface Visualization

This notebook provides tools to visualize `.ply` surfaces of neurons as rotating 3D objects.

In [ ]:
%load_ext autoreload
%autoreload 2

First, activate your environment in the terminal (the same you choose as the kernel for this notebook), ex. `conda ativate snakemake`

Install the following  
`conda install imageio ipywidgets ipykernel tqdm`  
`pip install pyvista[jupyter] plyfile `

In [ ]:
import pyvista as pv
from pathlib import Path
from tqdm import tqdm
from IPython.display import display, Image, HTML
from multiprocessing import Pool, cpu_count

import sys; sys.path.insert(0, '..')
from scripts.visualize_neurons import get_cell_ply_files, visualize_interactive_pyvista, create_rotation_gif

In [ ]:
# Path to the cell surfaces
surfaces_directory = "../emimesh/results/cells/"

In [ ]:

# Assumes surface_directory/cell_0, surface_directory/cell_1, ...
# surface_paths = [Path(p) for p in get_cell_ply_files(surfaces_directory)]
# surface_paths = sorted(surface_paths, key=lambda x: int(x.parent.parent.name.split("_")[1]))

# Assumes surface_directory/cell_type/cell_0, surface_directory/cell_type/cell_1, ...
surface_paths = [Path(p) for p in get_cell_ply_files(surfaces_directory)]
surface_paths = sorted(surface_paths, key=lambda x: int(x.parent.parent.name.split("_")[-1]))


print(f"Found {len(surface_paths)} cell surfaces.")
print(surface_paths[:3])

## GIF Generator

This function creates a rotating 3D animation of the surface and saves it as a GIF.

In [ ]:
# Generate GIF for the first neuron
gif_path = create_rotation_gif(surface_paths[0], frames_per_degree=0.5, verbose=True)

# Show the generated GIF
display(Image(filename=gif_path))

### Batch Processing

Loop through all neurons in the results folder.

In [ ]:
print(f"Using {cpu_count()//2} CPU cores for parallel processing.")
with Pool(processes=cpu_count()//2) as pool:
    results = list(tqdm(pool.imap(create_rotation_gif, surface_paths), total=len(surface_paths)))

print(f"Generated GIFs for {len(results)} neuron surfaces.")

## Display all cells

In [ ]:
max_N = 16

# Collect all surface.gif paths
gif_paths = [str(p.parent / "surface.gif") for p in surface_paths if (p.parent / "surface.gif").exists()]
neuron_labels = {path: p.parent.parent.name for path, p in zip(gif_paths, surface_paths)}

# Build the HTML grid
html_str = '<div style="display: grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap: 15px; text-align: center;">'
for path in gif_paths[:max_N]:
    label = neuron_labels[path]
    html_str += f'<div style="border: 1px solid #ddd; padding: 5px; border-radius: 5px;">'
    html_str += f'<img src="{path}" style="width: 100%; height: auto;"><br>'
    html_str += f'<strong>{label}</strong>'
    html_str += '</div>'
html_str += '</div>'

HTML(html_str)

## Display one cell surface

In [ ]:
pv.set_jupyter_backend('trame') # Ensures interactive plots work in Jupyter

# Example: Visualize the first cell found
visualize_interactive_pyvista(surface_paths[0])